# 台灣華語 Whisper small＋LoRA：Google Colab GPU

先以免費 GPU 配額嘗試；Google 不保證提供 GPU、特定型號或連線時長。這個 Notebook 不建立付費雲端機器、不發布模型。雲端 GPU 執行訓練，Cloudflare 保留網站／房間服務。

目前資料仍在下載／核對階段，**不代表已有可訓練資料**。每個階段分開執行，`RUN_TRAINING` 預設 False。訓練入口會重新做資料與 GPU 檢查，未通過不訓練。請先從「執行階段 → 變更執行階段類型」選擇 GPU。


In [ ]:
from pathlib import Path
import subprocess, sys

def run_step(command, cwd=None, require_success=True):
    """Stream child output into saved notebook outputs and keep a copy for diagnostics."""
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding='utf-8', errors='replace', bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    process.stdout.close()
    status = process.wait()
    result = subprocess.CompletedProcess(command, status, stdout=''.join(lines))
    if require_success and status:
        raise RuntimeError(f'步驟失敗（exit {status}），詳細原因已保存在上方輸出。')
    return result

REPO = Path('/content/Meeting-Live-Captions')
REPO_REF = 'main'  # 重跑既有run時改為run-lock記錄的commit
if not REPO.exists():
    run_step(['git','clone','https://github.com/timo893131/Meeting-Live-Captions.git',str(REPO)])
else:
    remote = subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()
    if remote.rstrip('/').removesuffix('.git') != 'https://github.com/timo893131/Meeting-Live-Captions':
        raise RuntimeError('此路徑不是預期的專案，已停止以保留既有內容。')
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('專案有本地修改，請先保存；沒有覆寫你的檔案。')
# A checkout of an already-existing main branch does not fetch newer fixes.
run_step(['git','-C',str(REPO),'fetch','origin',REPO_REF])
run_step(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'])
print('程式版本：',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
DATA_READY = False
TRAINING_IMPORTS_OK = False
_ = run_step([sys.executable,'-m','pip','install','-r',str(REPO/'training/requirements-colab.txt')])


In [ ]:
import torch
import importlib.metadata
TRAINING_IMPORTS_OK = False
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('尚無 CUDA GPU，請切換 Colab GPU 執行階段；未開始訓練。')
print('GPU:',torch.cuda.get_device_name(0))
print('GPU 記憶體 GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
probe = torch.ones((16,16),device='cuda')
assert torch.matmul(probe,probe)[0,0].item() == 16
print('CUDA 核心計算檢查通過（不是語音模型測試）。')
del probe
torch.cuda.empty_cache()
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
import accelerate
for package in ['transformers','peft','accelerate','huggingface-hub']:
    print(package,importlib.metadata.version(package))
TRAINING_IMPORTS_OK = True
print('訓練套件匯入通過；尚未載入模型或訓練。')


## 私人資料與輸出位置

掛載 Drive 由你在 Colab 完成。上傳可訓練的 WAV 與三份人工審核 manifest，音檔路徑相對 DATA_ROOT。保持原音，使用 16 kHz、mono、PCM16、5–30 秒工作副本。

現有爬蟲的 pending／unassigned manifest **不能直接用於訓練**。格式與必填審核欄位見 repo 的 docs/cloud-training.md。資料權利是 training（train）或 internal_evaluation（validation/test），不能只用「公開可讀」取代。未知講員預設擋下；若研究設定允許，須明確修改設定並揭露隔離限制。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json
from datetime import datetime, timezone
WORKFLOW_STAGE = 'collect'  # 先蒐集與核對；正式資料備妥後才改成'train'
if WORKFLOW_STAGE not in ['collect','train']:
    raise ValueError('WORKFLOW_STAGE只可設為collect或train')
DATA_ROOT = Path('/content/drive/MyDrive/meeting-live-captions/data')
PILOT_ROOT = DATA_ROOT.parent/'pilots'/'taiwan-mandarin-v1'
RUN_ID = datetime.now(timezone.utc).strftime('whisper-small-smoke-%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/content/drive/MyDrive/meeting-live-captions/runs')/RUN_ID
CONFIG = json.loads((REPO/'training/cloud-whisper-lora.json').read_text())
CONFIG['data']['root'] = str(DATA_ROOT)
CONFIG['training']['mode'] = 'smoke'  # 20 步，只驗證流程；pilot 也需獨立版本與實測評估
CONFIG_PATH = Path('/content/cloud-training-config.json')
CONFIG_PATH.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2))
print('階段：',WORKFLOW_STAGE)
print('小樣本：',PILOT_ROOT)
print('正式訓練資料：',DATA_ROOT,'訓練輸出：',RUN_DIR)


## 先收集來源，暫停不存在的FLEURS台灣子集

已由實際官方匯出清單確認，目前設定的cmn_hant_tw子集不可用。不要繼續重跑此下載格。請先使用[獨立CPU爬蟲](https://colab.research.google.com/github/timo893131/Meeting-Live-Captions/blob/main/notebooks/crawl_church_sources.ipynb)收集指定教會資源的文字及音訊索引，存到Drive待審；不會直接啟動訓練。


In [ ]:
RUN_COLLECTION = False  # 已確認FLEURS未提供設定的台灣子集，停止重試
if WORKFLOW_STAGE == 'collect' and RUN_COLLECTION:
    result = run_step([sys.executable,str(REPO/'scripts/tw_speech_pilot.py'),'--root',str(PILOT_ROOT),'--prepare'],cwd=REPO,require_success=False)
    collection_report = PILOT_ROOT/'reports/collection-report.json'
    if collection_report.exists():
        print('資料下載報告（已保存在Drive）：')
        print(collection_report.read_text())
    if result.returncode:
        print('來源檢查或下載未完成，請保留上方報告；沒有開始訓練。')
    else:
        print('小樣本已收集，下一步核對音訊／文字與去重。尚未產生正式train/validation/test。')
elif WORKFLOW_STAGE == 'collect':
    print('此來源已停用。請先執行 notebooks/crawl_church_sources.ipynb；不要打開此下載開關。')
else:
    print('目前是訓練階段，不重複收集小樣本。')


## 正式訓練資料檢查（train階段才執行）

collect階段只收集與核對，不會因缺少正式manifest而報錯。當資料已完成權利、聽稿、近似去重與分組後，將上方WORKFLOW_STAGE改為train再檢查。下載小樣本不會自動建立正式train/validation/test。

同一recording、speaker、原音／PCM hash或相同文字不可跨集合。test只作資料洩漏審計，不傳給Trainer或early stopping。


In [ ]:
DATA_READY = False
if WORKFLOW_STAGE == 'train':
    result = run_step([sys.executable,str(REPO/'scripts/cloud_training_data.py'),'--config',str(CONFIG_PATH)],cwd=REPO,require_success=False)
    try:
        data_report = json.loads(result.stdout)
        DATA_READY = result.returncode == 0 and data_report.get('status') == 'passed'
    except json.JSONDecodeError:
        print('資料檢查未回傳有效JSON，請保留上方完整診斷。')
    if not DATA_READY:
        print('正式資料尚未通過檢查，訓練保持關閉。')
else:
    print('目前為資料準備階段，跳過正式訓練manifest檢查。完成核對與分組後再切換train。')


## 小規模試訓練與中斷續跑

預設 LoRA r=8、batch=1、gradient accumulation=4、20 steps、每10步評估及存checkpoint。30分鐘為每次 train 呼叫的逐步檢查上限，不包含前置下載／資料檢查／基線與後測評估。先驗證流程，再決定正式步數；20步不保證品質提升。

`RUN_TRAINING=True` 才會呼叫訓練。首次解析模型 commit 後寫入 run-lock.json，resume 必須使用相同程式、資料hash、設定及主要套件版本。若中斷，將 RUN_DIR 指向原目錄並設定 RESUME=True；沒有 checkpoint 時需使用新 run。成功完成的 run 不覆寫。


In [ ]:
RUN_TRAINING = False
RESUME = False
if WORKFLOW_STAGE != 'train':
    print('目前為資料準備階段，沒有執行訓練。')
elif not RUN_TRAINING:
    print('尚未開始訓練。資料及GPU套件檢查通過後，再明確啟用RUN_TRAINING。')
elif not globals().get('DATA_READY',False) or not globals().get('TRAINING_IMPORTS_OK',False):
    print('尚未通過資料或GPU套件檢查，未執行訓練命令。')
else:
    command=[sys.executable,str(REPO/'scripts/train_cloud_whisper.py'),'--config',str(CONFIG_PATH),'--run-dir',str(RUN_DIR)]
    if RESUME:
        command.append('--resume')
    run_step(command,cwd=REPO)


## 讀取結果

checkpoint、adapter、processor 與驗證結果保存於 Drive。before/after CER 採同一個雲端正規化版本；它是 validation 指標，不能當作未見test或真實即時延遲。最終還需選定候選後獨立跑 test、正常速度音訊／麥克風測試，並檢查轉回 CTranslate2/int8 的結果。

前次上傳紀錄已確認 CUDA 小運算與套件匯入；請仍以本次執行輸出為準。**尚未在 Colab GPU 執行微調**，沒有 checkpoint 或訓練改善結果，當前來源沒有正式 train-ready 資料。


In [ ]:
if WORKFLOW_STAGE == 'collect':
    report=PILOT_ROOT/'reports/collection-report.json'
    if report.exists():
        data=json.loads(report.read_text())
        print(json.dumps({key:data.get(key) for key in ['status','audio_downloaded','downloaded_hours','usable_paired_hours','pending_review_hours','failures']},ensure_ascii=False,indent=2))
        print('完整報告：',report)
    else:
        print('尚無收集報告，請先執行上方資料準備格。')
else:
    report=RUN_DIR/'final-report.json'
    if report.exists():
        print(report.read_text())
    else:
        print('尚無完成報告。若已試跑，可查看RUN_DIR/checkpoints與attempts。')
